In [33]:
import pandas as pd 

In [34]:
df = pd.read_csv("stage0_scanner_audit.csv")

In [35]:
df["max_auroc"].mean()

np.float64(0.5037842437013751)

In [36]:
df = pd.read_csv("stage1_frequency.csv")

freq_cols = ["freq_chexpert", "freq_rsna", "freq_siim_covid", "freq_tb"]
FREQ_THR  = 0.01   # >=1% per methodology
MIN_DS    = 2      # active on >=2 datasets

# Number of datasets each latent is active on (>=1%).
df["n_datasets_active"] = (df[freq_cols] >= FREQ_THR).sum(axis=1)

# Sanity check: is_l1 should equal "active on >=MIN_DS datasets".
assert ((df["n_datasets_active"] >= MIN_DS).astype(int) == df["is_l1"]).all()

passed_L1_features = df[df["is_l1"] == 1].copy()
print(f"L1-passing features: {len(passed_L1_features)} / {len(df)}")


L1-passing features: 863 / 4096


In [37]:
# Average activation frequency per dataset, restricted to L1-passing latents.
means_pct = passed_L1_features[freq_cols].mean() * 100
print("Average activation frequency on L1-passing latents (%):")
for c in freq_cols:
    print(f"  {c:18s} {means_pct[c]:6.3f}")


Average activation frequency on L1-passing latents (%):
  freq_chexpert       4.439
  freq_rsna           4.524
  freq_siim_covid     4.415
  freq_tb             3.720


In [38]:
# Distribution of L1-passing latents by number of datasets they fire on.
counts = passed_L1_features["n_datasets_active"].value_counts().sort_index()
total  = len(passed_L1_features)

print("Active on exactly N datasets:")
for n, c in counts.items():
    print(f"  N={n}: {c:4d} ({c/total*100:5.1f}%)")

print("\nActive on at least N datasets:")
for n in [2, 3, 4]:
    c = (passed_L1_features["n_datasets_active"] >= n).sum()
    print(f"  >={n}: {c:4d} ({c/total*100:5.1f}%)")


Active on exactly N datasets:
  N=2:  229 ( 26.5%)
  N=3:  131 ( 15.2%)
  N=4:  503 ( 58.3%)

Active on at least N datasets:
  >=2:  863 (100.0%)
  >=3:  634 ( 73.5%)
  >=4:  503 ( 58.3%)


In [44]:
l2 = pd.read_csv("stage2_semantic.csv")
l2_succ = l2[l2["is_l2"]==1]
len(l2_succ)

279